# Un même PIB, trois vitesses de lecture · *One GDP, three reading speeds*

Notebook compagnon du chapitre **9. Comprendre le PIB : la mesure de la richesse produite par un pays** — [lire l'article](https://nmlab.io/ressources/comprendre-le-pib).
Companion notebook to chapter **9. Understanding GDP: The Measure of the Wealth a Country Produces** — [read the article](https://nmlab.io/en/ressources/understanding-gdp).

**Exécutez l'unique cellule ci-dessous** (bouton ▶ ou Ctrl+Entrée) : la figure se régénère avec les **données FRED du jour**. Passez `LANG = "en"` en tête de cellule pour les libellés anglais. — Run the single cell below (▶ or Ctrl+Enter) to rebuild the figure with **today's FRED data**; set `LANG = "en"` at the top for English labels.

Code : licence MIT · © 2026 [NMLab](https://nmlab.io) · dépôt [nmlab-finance/nmlab-figures](https://github.com/nmlab-finance/nmlab-figures)

In [ ]:
LANG = "fr"   # "fr" ou "en" — langue des libellés / label language

# Récupère puis active le style partagé NMLab (thème sombre + police Inter).
# Fetch and activate the shared NMLab style (dark theme + Inter font).
import urllib.request

urllib.request.urlretrieve("https://raw.githubusercontent.com/nmlab-finance/nmlab-figures/main/nmlab_style.py", "nmlab_style.py")
import nmlab_style as nm

nm.setup()


from pandas import DataFrame

def load_readings() -> DataFrame:
    """Les cinq derniers trimestres du PIB réel (GDPC1) lus de trois façons, en % (FRED, direct).
    The last five quarters of real GDP read three ways, in %, live from FRED."""
    gdp = nm.load_fred("GDPC1")
    qoq = gdp.pct_change(1) * 100
    annualized = ((1 + gdp.pct_change(1)) ** 4 - 1) * 100
    yoy = gdp.pct_change(4) * 100
    table = DataFrame({"qoq": qoq, "annualized": annualized, "yoy": yoy}).dropna().tail(5)
    return table

readings = load_readings()


import numpy as np
from matplotlib.figure import Figure

LABELS = {
    "fr": dict(
        title="Un même PIB, trois vitesses de lecture",
        sub="PIB réel américain : la même réalité, trois conventions d'affichage",
        ylab="%",
        legend=["variation t/t", "rythme annualisé", "glissement sur un an"],
        note="Le rythme annualisé ≈ la variation trimestrielle composée sur quatre trimestres (convention américaine).\n"
             "Source : BEA via FRED (GDPC1) — millésime du 15 juillet 2026."),
    "en": dict(
        title="One GDP, three reading speeds",
        sub="U.S. real GDP: the same reality under three display conventions",
        ylab="%",
        legend=["quarter-on-quarter", "annualized pace", "year-over-year"],
        note="The annualized pace ≈ the quarterly change compounded over four quarters (U.S. convention).\n"
             "Source: BEA via FRED (GDPC1) — vintage of July 15, 2026."),
}

def quarter_labels(index, lang: str) -> list[str]:
    """Étiquettes de trimestre localisées (T1 2025 / Q1 2025)."""
    prefix = "T" if lang == "fr" else "Q"
    return [f"{prefix}{(d.month - 1) // 3 + 1} {d.year}" for d in index]

def num(value: float, lang: str) -> str:
    """Nombre à une décimale, signe « − », virgule décimale en français."""
    sign = "−" if value < 0 else ""
    body = f"{abs(value):.1f}".replace(".", ",") if lang == "fr" else f"{abs(value):.1f}"
    return f"{sign}{body}"

def build_figure(readings: "DataFrame", lang: str) -> Figure:
    """Barres groupées par trimestre : variation t/t, rythme annualisé, glissement annuel."""
    text = LABELS[lang]
    light = "#a7c4e2"
    fig = nm.figure(height_px=1026)
    ax = nm.axes(fig, bottom=0.14)
    ax.grid(axis="x", visible=False)
    x = np.arange(len(readings))
    width = 0.27
    series = [("qoq", light), ("annualized", nm.COLORS["blue"]), ("yoy", nm.COLORS["rose"])]
    for (col, colour), label, offset in zip(series, text["legend"], (-width, 0, width)):
        values = readings[col].values
        ax.bar(x + offset, values, width, color=colour, label=label, zorder=3)
        for xi, v in zip(x + offset, values):
            above = v >= 0
            ax.annotate(num(v, lang), (xi, v), xytext=(0, 8 if above else -8),
                        textcoords="offset points", ha="center",
                        va="bottom" if above else "top", fontsize=20, fontweight="bold",
                        color=nm.COLORS["text"] if above else nm.COLORS["rose"])
    ax.axhline(0, color=nm.COLORS["muted"], linewidth=1.4, alpha=0.9)
    ax.set_ylim(-1.4, 5.4)
    ax.set_yticks(range(-1, 6))
    ax.set_ylabel(text["ylab"])
    ax.set_xlim(-0.6, len(readings) - 0.4)
    ax.set_xticks(x)
    ax.set_xticklabels(quarter_labels(readings.index, lang), fontsize=23, color=nm.COLORS["muted"])
    ax.tick_params(axis="x", length=0)
    ax.legend(loc="upper left", frameon=False, fontsize=23, labelcolor=nm.COLORS["text"],
              handlelength=1.1, handleheight=1.1, borderaxespad=0.9)
    nm.header(fig, text["title"], text["sub"])
    nm.footer(fig, text["note"])
    return fig

build_figure(readings, LANG)